[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Fixtures


## What you will be able to do

Move the arrange step of a test into a fixture, a function marked with `@pytest.fixture` that pytest
runs for every test that names it as an argument. Clean up after a test with `yield`, share fixtures
between test files with `conftest.py`, choose how long a fixture's value lasts with its scope, and
use pytest's own fixtures for temporary folders, printed output and environment variables.


## The idea

### The problem

The **Test Structure** notebook gave its summary tests a helper, `tuesday`, which every test called.
That works until the tests need more than a list. A test of the summary script needs a file of
readings on disk, which it has to write before it runs and remove after it, and a test that fails at
its `assert` never reaches the line that removes the file, so the next run finds it still there. A
helper written in one test file is not available in another, so the **Test Structure** notebook's
solutions copied one. And some setup is slow, such as loading a catalog of stations, which a hundred
tests could share instead of repeating.

Every one of those is setup that belongs to no single test, and pytest has one mechanism for all of
them.

### What a fixture is

> A **fixture** is a function marked with `@pytest.fixture` that prepares something a test needs. A
> test **requests** a fixture by naming it as an argument, and pytest runs the fixture before the
> test and passes the test what the fixture returned. A fixture that uses `yield` instead of `return`
> hands its value over at the `yield`, and runs the code after it once the test is over, whether the
> test passed or failed, which is the test's cleanup. A fixture's **scope** sets how long its value
> lasts: a new value for every test, by default, or one value for every test in a class, a module, a
> package or the whole run. Fixtures in a file named **`conftest.py`** are available to every test in
> that folder and the folders below it.

### Why it works that way

- **An argument's name is the request.** pytest matches the names of a test's arguments to the names
  of fixtures, so a test lists what it needs in its first line and calls nothing. This is the
  registering the **Decorators** notebook described: `@pytest.fixture` marks a function so that
  pytest can find it later.
- **A fixture runs again for every test that asks.** With the default scope, every test gets a value
  of its own, so the tests stay independent, as the **Test Structure** notebook asked.
- **`yield` keeps setup and cleanup in one function.** The code before the `yield` arranges, and the
  code after it cleans up, and pytest runs the cleanup even when the test failed.
- **A wider scope trades independence for time.** A fixture scoped to a module runs once for every
  test in the file, and a test that changes its value changes it for every test after it.
- **Fixtures can request fixtures.** A fixture names other fixtures as its arguments, as a test does,
  and pytest sets them up in order and tears them down in reverse.
- **pytest brings fixtures of its own.** `tmp_path` is a new, empty folder for every test, `capsys`
  captures what the code under test prints, and `monkeypatch` changes an environment variable for one
  test and puts it back afterward.

### Where you will meet this

pytest's documentation on fixtures covers requesting them, `yield` fixtures, scopes and
`conftest.py`, and its list of built-in fixtures includes `tmp_path`, `capsys` and `monkeypatch`,
which this notebook uses. A plugin for pytest often adds fixtures of its own, which a test requests
by name in the same way. The **Parametrize** notebook gives a fixture a list of values, so that every
test that requests it runs once for each value.

### What this notebook covers

- A fixture, and a test that requests it by name
- `--setup-show`, which shows every fixture set up and torn down
- A fixture that requests another fixture: a file of readings in `tmp_path`
- Cleanup with `yield`, which runs after a failed test too
- `conftest.py`: fixtures shared by every test in a folder
- Scope: a value for every test, or one shared by a class, a module or the whole run
- `capsys` for printed output, and `monkeypatch` for an environment variable
- The summary script tested three ways, with fixtures doing all the arranging
- Four errors: a fixture called directly, a misspelled fixture, a scope mismatch, and a shared value
  a test changes

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import sys
from pathlib import Path

Path("test_first_look.py").write_text('''
import pytest


@pytest.fixture
def readings():
    return [4.2, None, 5.8]


def test_three_readings(readings):
    assert len(readings) == 3


def test_one_reading_is_missing(readings):
    assert readings.count(None) == 1
''')

finished = subprocess.run([sys.executable, "-m", "pytest", "-q", "--setup-show"],
                          capture_output=True, text=True)
print(finished.stdout.strip())
```

```
SETUP    F readings
        test_first_look.py::test_three_readings (fixtures used: readings).
        TEARDOWN F readings
        SETUP    F readings
        test_first_look.py::test_one_reading_is_missing (fixtures used: readings).
        TEARDOWN F readings
2 passed in 0.01s
```

Both tests name `readings` as an argument, and neither calls it. pytest found the function marked
`@pytest.fixture` with that name, ran it before each test, and passed in the list it returned.
`--setup-show` prints every fixture as pytest sets it up and tears it down, and the `F` says the
fixture's scope is a function: every test got a list of its own.


## Setup

Six imports, and the functions that run pytest.

- `subprocess` runs pytest as a program of its own, in `run_pytest`
- `sys` names the Python that runs it
- `os` passes pytest the environment, with `NO_COLOR` and `PYTHONDONTWRITEBYTECODE` set in it, as in
  the **Your First Test** notebook
- `re` takes out of pytest's report the parts that differ between computers
- `Path` makes the project's folders, and checks what the tests leave behind
- `shutil` removes the scratch folder at the end

`pytest_report` and `run_pytest` are the functions the **Your First Test** notebook wrote, which run
`python -m pytest` in the project's folder, `scratch/stations`, and that notebook explains each of
their settings.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


## Worked examples

### A fixture, and a test that requests it

The project is the one the **Test Structure** notebook finished with, and its module is unchanged:


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


The helper `tuesday` becomes a fixture: the same function with `@pytest.fixture` above it. The tests
stop calling it, and name it as an argument instead:


In [3]:
%%writefile scratch/stations/tests/test_summary.py
import pytest

from readings import summarize


@pytest.fixture
def tuesday():
    """Tuesday's lines of readings, as the stations sent them."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


def test_summary_of_tuesday(tuesday):
    summary = summarize(tuesday)

    assert summary == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_a_blank_line_changes_nothing(tuesday):
    tuesday.insert(3, "")

    summary = summarize(tuesday)

    assert summary == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_tuesday_has_ten_lines(tuesday):
    assert len(tuesday) == 10


Writing scratch/stations/tests/test_summary.py


In [4]:
run_pytest("tests/test_summary.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_summary.py::test_summary_of_tuesday PASSED                    [ 33%]
tests/test_summary.py::test_a_blank_line_changes_nothing PASSED          [ 66%]
tests/test_summary.py::test_tuesday_has_ten_lines PASSED                 [100%]

============================== 3 passed ===============================


Inside a test, `tuesday` is the list the fixture returned, not the function: pytest called the
function and passed in the result. The test `test_a_blank_line_changes_nothing` inserted a line into
its list, and `test_tuesday_has_ten_lines`, which ran after it, still found ten lines, because every
test received a list of its own.

### --setup-show: fixtures set up and torn down

`--setup-show` prints what pytest does around every test: `SETUP` when it runs a fixture, `TEARDOWN`
when it finishes with the fixture's value, and the fixtures each test used:


In [5]:
run_pytest("tests/test_summary.py", "--setup-show", "-q")



        SETUP    F tuesday
        tests/test_summary.py::test_summary_of_tuesday (fixtures used: tuesday).
        TEARDOWN F tuesday
        SETUP    F tuesday
        tests/test_summary.py::test_a_blank_line_changes_nothing (fixtures used: tuesday).
        TEARDOWN F tuesday
        SETUP    F tuesday
        tests/test_summary.py::test_tuesday_has_ten_lines (fixtures used: tuesday).
        TEARDOWN F tuesday
3 passed


`F` is the fixture's scope, a function, which is why every test has a `SETUP` of its own. The report
grows with the suite, so `--setup-show` is for the moments when you need to know which fixtures a
test used, and when.

### A fixture that requests a fixture: a file in tmp_path

The summary script reads a file, so its tests need one. `tmp_path` is a fixture pytest provides: a
new, empty folder for every test, somewhere outside the project, which pytest creates before the
test. A fixture requests it the way a test does, by naming it, and here `tuesday_file` also requests
`tuesday`, to write its lines into a file in that folder:


In [6]:
%%writefile scratch/stations/tests/test_files.py
import pytest

from readings import summarize


@pytest.fixture
def tuesday():
    """Tuesday's lines of readings, as the stations sent them."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


@pytest.fixture
def tuesday_file(tmp_path, tuesday):
    """Tuesday's readings, in a file of their own."""
    path = tmp_path / "tuesday.csv"
    path.write_text("\n".join(tuesday) + "\n", encoding="utf-8")
    return path


def test_a_file_of_readings_summarizes(tuesday_file):
    with open(tuesday_file, encoding="utf-8") as file:
        summary = summarize(file)

    assert summary == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_the_folder_holds_only_this_file(tuesday_file):
    assert [path.name for path in tuesday_file.parent.iterdir()] == ["tuesday.csv"]


Writing scratch/stations/tests/test_files.py


In [7]:
run_pytest("tests/test_files.py", "--setup-show", "-q")



SETUP    S tmp_path_factory
        SETUP    F tmp_path (fixtures used: tmp_path_factory)
        SETUP    F tuesday
        SETUP    F tuesday_file (fixtures used: tmp_path, tuesday)
        tests/test_files.py::test_a_file_of_readings_summarizes (fixtures used: request, tmp_path, tmp_path_factory, tuesday, tuesday_file).
        TEARDOWN F tuesday_file
        TEARDOWN F tuesday
        TEARDOWN F tmp_path
        SETUP    F tmp_path (fixtures used: tmp_path_factory)
        SETUP    F tuesday
        SETUP    F tuesday_file (fixtures used: tmp_path, tuesday)
        tests/test_files.py::test_the_folder_holds_only_this_file (fixtures used: request, tmp_path, tmp_path_factory, tuesday, tuesday_file).
        TEARDOWN F tuesday_file
        TEARDOWN F tuesday
        TEARDOWN F tmp_path
TEARDOWN S tmp_path_factory
2 passed


pytest worked out the order from the arguments: `tmp_path` and `tuesday` before `tuesday_file`, which
needs both, and the teardowns in reverse. `tmp_path_factory`, with `S` for session, is the fixture
behind `tmp_path`, which makes one folder for the run and a new folder inside it for every test, and
the second test found its folder holding one file. The folder's full path contains the name of the
user running the tests, which is one reason these notebooks never print it.

### Cleanup with yield

Some setup has to be undone. A fixture that writes a file into the folder the tests run in has to
remove it afterward, or the next run finds it there. A fixture that uses `yield` hands its value over
at the `yield`, and pytest runs the lines after the `yield` when the test is over. Here, the second
test fails on purpose:


In [8]:
%%writefile scratch/stations/tests/test_cleanup.py
from pathlib import Path

import pytest

from readings import summarize


@pytest.fixture
def bergen_file():
    """A file of Bergen's readings in the folder the tests run in, removed after the test."""
    path = Path("bergen.csv")
    path.write_text("Bergen,4.2\nBergen,5.8\n", encoding="utf-8")
    yield path
    path.unlink()


def test_bergen_summarizes(bergen_file):
    with open(bergen_file, encoding="utf-8") as file:
        assert summarize(file) == {"Bergen": 5.0}


def test_bergen_has_the_wrong_mean(bergen_file):
    with open(bergen_file, encoding="utf-8") as file:
        assert summarize(file) == {"Bergen": 4.2}


Writing scratch/stations/tests/test_cleanup.py


In [9]:
run_pytest("tests/test_cleanup.py", "--setup-show", "-q")

print("\nbergen.csv left in the project:", (PROJECT / "bergen.csv").exists())



        SETUP    F bergen_file
        tests/test_cleanup.py::test_bergen_summarizes (fixtures used: bergen_file).
        TEARDOWN F bergen_file
        SETUP    F bergen_file
        tests/test_cleanup.py::test_bergen_has_the_wrong_mean (fixtures used: bergen_file)F
        TEARDOWN F bergen_file
=================================== FAILURES ===================================
________________________ test_bergen_has_the_wrong_mean ________________________

bergen_file = PosixPath('bergen.csv')

    def test_bergen_has_the_wrong_mean(bergen_file):
        with open(bergen_file, encoding="utf-8") as file:
>           assert summarize(file) == {"Bergen": 4.2}
E           AssertionError: assert {'Bergen': 5.0} == {'Bergen': 4.2}
E             
E             Differing items:
E             {'Bergen': 5.0} != {'Bergen': 4.2}
E             Use -v to get more diff

tests/test_cleanup.py:24: AssertionError
=========================== short test summary info ============================
FAILED

The second test failed at its `assert`, and `TEARDOWN F bergen_file` still followed it, so the file
is gone. If the test had removed the file itself, on a line after its `assert`, the failure would
have stopped the test before that line, and the file would have stayed. Code after the `yield` runs
however the test ends. Writing into `tmp_path` needs no cleanup at all, which is why a test that
needs a file usually uses it.

The test that failed on purpose has made its point, so its file goes:


In [10]:
(PROJECT / "tests" / "test_cleanup.py").unlink()

print(sorted(path.name for path in (PROJECT / "tests").iterdir()))


['test_files.py', 'test_summary.py']


### conftest.py: fixtures for every test in a folder

`tuesday` is now defined twice, in `test_summary.py` and in `test_files.py`. A fixture in a file
named `conftest.py` is available to every test in that file's folder and the folders below it,
without an import, so the two fixtures move there, and the test files keep only their tests:


In [11]:
%%writefile scratch/stations/tests/conftest.py
import pytest


@pytest.fixture
def tuesday():
    """Tuesday's lines of readings, as the stations sent them."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


@pytest.fixture
def tuesday_file(tmp_path, tuesday):
    """Tuesday's readings, in a file of their own."""
    path = tmp_path / "tuesday.csv"
    path.write_text("\n".join(tuesday) + "\n", encoding="utf-8")
    return path


Writing scratch/stations/tests/conftest.py


In [12]:
for name in ["test_summary.py", "test_files.py"]:
    path = PROJECT / "tests" / name
    source = path.read_text()
    start = source.index("@pytest.fixture")
    end = source.index("def test_")
    path.write_text(source[:start].replace("import pytest\n\n", "") + source[end:])

print((PROJECT / "tests" / "test_files.py").read_text())


from readings import summarize


def test_a_file_of_readings_summarizes(tuesday_file):
    with open(tuesday_file, encoding="utf-8") as file:
        summary = summarize(file)

    assert summary == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_the_folder_holds_only_this_file(tuesday_file):
    assert [path.name for path in tuesday_file.parent.iterdir()] == ["tuesday.csv"]



In [13]:
run_pytest("tests", "-q")


.....                                                                    [100%]
5 passed


The tests still found both fixtures, now from `conftest.py`. `--fixtures` lists every fixture a test
in a folder can request, pytest's own first, and then the ones from `conftest.py`, with their
docstrings. The cell prints the part from `conftest.py`:


In [14]:
report = pytest_report("--fixtures", "tests", "-q")

print(report[report.rindex("\n", 0, report.index("fixtures defined from conftest")) + 1:])


------------------------ fixtures defined from conftest ------------------------
tuesday_file -- tests/conftest.py:12
    Tuesday's readings, in a file of their own.

tuesday -- tests/conftest.py:5
    Tuesday's lines of readings, as the stations sent them.


no tests ran


Nothing imports `conftest.py`. pytest reads it before it collects the tests in its folder, and its
fixtures reach every test in `tests` and the folders below, and no test outside them.

### Scope: how long a value lasts

A fixture's scope is how long pytest keeps its value. Here is a catalog of where the stations are,
which is quick to build, standing in for setup that is slow, such as connecting to a database. With
`scope="module"`, pytest builds it once for all the tests in the file:


In [15]:
%%writefile scratch/stations/tests/test_catalog.py
import pytest


@pytest.fixture(scope="module")
def catalog():
    """Where the stations are, as (latitude, longitude), built once for every test in this file."""
    return {"Bergen": (60.39, 5.32), "Oslo": (59.91, 10.75), "Svalbard": (78.22, 15.65), "Tromso": (69.65, 18.96)}


def test_every_station_is_in_the_catalog(catalog):
    assert sorted(catalog) == ["Bergen", "Oslo", "Svalbard", "Tromso"]


def test_svalbard_is_the_northernmost(catalog):
    assert max(catalog, key=lambda station: catalog[station][0]) == "Svalbard"


def test_oslo_is_the_southernmost(catalog):
    assert min(catalog, key=lambda station: catalog[station][0]) == "Oslo"


Writing scratch/stations/tests/test_catalog.py


In [16]:
run_pytest("tests/test_catalog.py", "--setup-show", "-q")



    SETUP    M catalog
        tests/test_catalog.py::test_every_station_is_in_the_catalog (fixtures used: catalog).
        tests/test_catalog.py::test_svalbard_is_the_northernmost (fixtures used: catalog).
        tests/test_catalog.py::test_oslo_is_the_southernmost (fixtures used: catalog).
    TEARDOWN M catalog
3 passed


One `SETUP M catalog` before the three tests and one `TEARDOWN M catalog` after them, where a
function-scoped fixture would have been set up three times. pytest has five scopes:

| Scope | The value lasts for | Letter in `--setup-show` |
|---|---|---|
| `function`, the default | one test | `F` |
| `class` | every test in one class | `C` |
| `module` | every test in one file | `M` |
| `package` | every test in one package, a folder with an `__init__.py` | `P` |
| `session` | every test in the run | `S` |

A shared value is shared by every test that requests it, so a test must not change it, and Common
errors shows a test that does. A fixture can request a fixture of its own scope or a wider one, and
pytest refuses the other way around, which Common errors shows too.

### capsys and monkeypatch

The summary script prints its summary, and it takes the file's path as an argument or, when there is
none, from an environment variable, `STATIONS_READINGS`. Its work is in a function, `main`, which a
test can call:


In [17]:
%%writefile scratch/stations/summary.py
"""Print each station's mean temperature from a file of readings.

    python summary.py readings.csv

With no path given, the path comes from the environment variable STATIONS_READINGS.
"""

import os
import sys

from readings import summarize, to_fahrenheit


def main(arguments):
    path = arguments[0] if arguments else os.environ["STATIONS_READINGS"]
    with open(path, encoding="utf-8") as file:
        summary = summarize(file)
    for station, celsius in summary.items():
        if celsius is None:
            print(station, "no readings")
        else:
            print(station, f"{celsius:.1f} C, {to_fahrenheit(celsius):.1f} F")


if __name__ == "__main__":
    main(sys.argv[1:])


Writing scratch/stations/summary.py


Two more of pytest's fixtures test it. `capsys` captures what the code under test prints, and its
`readouterr()` returns the text printed so far, as `out` and `err`. `monkeypatch` changes things for
one test and puts them back afterward: `monkeypatch.setenv` sets an environment variable that
disappears when the test is over, so no other test sees it:


In [18]:
%%writefile scratch/stations/tests/test_script.py
from summary import main


def test_main_prints_every_station(tuesday_file, capsys):
    main([str(tuesday_file)])

    printed = capsys.readouterr().out
    assert printed.splitlines() == ["Bergen 5.0 C, 41.0 F", "Oslo -2.0 C, 28.4 F",
                                    "Svalbard no readings", "Tromso -6.0 C, 21.2 F"]


def test_main_reads_the_path_from_the_environment(tuesday_file, capsys, monkeypatch):
    monkeypatch.setenv("STATIONS_READINGS", str(tuesday_file))

    main([])

    assert capsys.readouterr().out.startswith("Bergen 5.0 C, 41.0 F\n")


Writing scratch/stations/tests/test_script.py


In [19]:
run_pytest("tests/test_script.py", "-v")


============================= test session starts ==============================
collecting ... collected 2 items

tests/test_script.py::test_main_prints_every_station PASSED              [ 50%]
tests/test_script.py::test_main_reads_the_path_from_the_environment PASSED [100%]

============================== 2 passed ===============================


The first test called `main` with the file's path, and compared what it printed with the lines the
**Why Test** notebook's end-to-end test expected. The second test gave `main` no path, so `main`
read the variable `monkeypatch` had set. Neither test needed to start a program, which keeps both
fast, and neither left the variable set for the next test.

### The summary script, tested three ways

The pieces of this notebook, in one file: the script tested as a function, through the environment,
and as a program started the way a person starts it. The fixtures do all the arranging, and none of
the tests writes a file, sets a variable, or cleans anything up itself:


In [20]:
%%writefile scratch/stations/tests/test_script.py
import subprocess
import sys

from summary import main

EXPECTED = ["Bergen 5.0 C, 41.0 F", "Oslo -2.0 C, 28.4 F", "Svalbard no readings", "Tromso -6.0 C, 21.2 F"]


def test_main_prints_every_station(tuesday_file, capsys):
    main([str(tuesday_file)])

    assert capsys.readouterr().out.splitlines() == EXPECTED


def test_main_reads_the_path_from_the_environment(tuesday_file, capsys, monkeypatch):
    monkeypatch.setenv("STATIONS_READINGS", str(tuesday_file))

    main([])

    assert capsys.readouterr().out.splitlines() == EXPECTED


def test_the_script_runs_as_a_program(tuesday_file, monkeypatch):
    monkeypatch.setenv("STATIONS_READINGS", str(tuesday_file))

    finished = subprocess.run([sys.executable, "summary.py"], capture_output=True, text=True)

    assert finished.stdout.splitlines() == EXPECTED


Overwriting scratch/stations/tests/test_script.py


In [21]:
run_pytest("tests/test_script.py", "--setup-show", "-q")
print()
run_pytest("-q")



SETUP    S tmp_path_factory
        SETUP    F tmp_path (fixtures used: tmp_path_factory)
        SETUP    F tuesday
        SETUP    F tuesday_file (fixtures used: tmp_path, tuesday)
        SETUP    F capsys
        tests/test_script.py::test_main_prints_every_station (fixtures used: capsys, request, tmp_path, tmp_path_factory, tuesday, tuesday_file).
        TEARDOWN F capsys
        TEARDOWN F tuesday_file
        TEARDOWN F tuesday
        TEARDOWN F tmp_path
        SETUP    F tmp_path (fixtures used: tmp_path_factory)
        SETUP    F tuesday
        SETUP    F tuesday_file (fixtures used: tmp_path, tuesday)
        SETUP    F capsys
        SETUP    F monkeypatch
        tests/test_script.py::test_main_reads_the_path_from_the_environment (fixtures used: capsys, monkeypatch, request, tmp_path, tmp_path_factory, tuesday, tuesday_file).
        TEARDOWN F monkeypatch
        TEARDOWN F capsys
        TEARDOWN F tuesday_file
        TEARDOWN F tuesday
        TEARDOWN F tmp_path

The program started by `subprocess.run` inherited the environment, variable and all, since
`monkeypatch.setenv` changed the environment of the process pytest runs in. When the test was over,
`TEARDOWN F monkeypatch` put the environment back.

### Where each part came from

| In the tests | What it relies on | The section that showed it |
|---|---|---|
| `tuesday_file` as an argument | a fixture requested by name | A fixture, and a test that requests it |
| `tuesday_file` built on `tmp_path` and `tuesday` | fixtures that request fixtures, in order | A fixture that requests a fixture: a file in tmp_path |
| no file left behind | a new folder for every test, and cleanup that runs anyway | Cleanup with yield |
| no import of `tuesday_file` | `conftest.py`, read by pytest for the whole folder | conftest.py: fixtures for every test in a folder |
| a file for every test, but one catalog for a file | the default scope, and a wider one | Scope: how long a value lasts |
| `capsys` and `monkeypatch` | pytest's own fixtures, which undo what they change | capsys and monkeypatch |
| the order of `SETUP` and `TEARDOWN` | what pytest does around a test | --setup-show: fixtures set up and torn down |

Every test in the file states what it needs in its first line, and does nothing else before its act
step.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/04-fixtures-solutions.ipynb).

**1.** Write `tests/test_oslo.py` with a fixture, `oslo_lines`, that returns two readings from Oslo,
`"Oslo,-2.4"` and `"Oslo,-1.6"`, and a test that requests it and checks that the summary is
`{"Oslo": -2.0}`. Run the file with `-v`.


In [22]:
# your code here


**2.** Add a second test to the file that requests `oslo_lines` and checks that there are two lines.
Run the file with `--setup-show`, and count how many times `oslo_lines` was set up.


In [23]:
# your code here


**3.** Add a fixture, `oslo_file`, that requests `oslo_lines` and `tmp_path` and returns the path of
a file holding the lines, and a test that summarizes the file. Run the file with `-q`.


In [24]:
# your code here


**4.** Move `oslo_lines` and `oslo_file` into `tests/conftest.py`, adding them to the fixtures
already there, and check that the tests in `tests/test_oslo.py` still pass.


In [25]:
# your code here


**5.** Write a test that calls `main` from `summary.py` with the path from `oslo_file`, and uses
`capsys` to check that it printed `Oslo -2.0 C, 28.4 F`. Run it.


In [26]:
# your code here


**6.** Write a test that uses `monkeypatch` to point `STATIONS_READINGS` at `oslo_file`, calls `main`
with no arguments, and checks the printed line. Run it, and then check in a notebook cell that
`STATIONS_READINGS` is not set in this notebook's own environment.


In [27]:
# your code here


## Common errors

### Fixture "bergen_lines" called directly


In [28]:
%%writefile scratch/stations/tests/test_direct.py
import pytest

from readings import summarize


@pytest.fixture
def bergen_lines():
    return ["Bergen,4.2", "Bergen,5.8"]


def test_bergen_summarizes():
    assert summarize(bergen_lines()) == {"Bergen": 5.0}


Writing scratch/stations/tests/test_direct.py


In [29]:
run_pytest("tests/test_direct.py", "-q")


F                                                                        [100%]
=================================== FAILURES ===================================
____________________________ test_bergen_summarizes ____________________________
Fixture "bergen_lines" called directly. Fixtures are not meant to be called directly,
but are created automatically when test functions request them as parameters.
See https://docs.pytest.org/en/stable/explanation/fixtures.html for more information about fixtures, and
https://docs.pytest.org/en/stable/deprecations.html#calling-fixtures-directly
=========================== short test summary info ============================
FAILED tests/test_direct.py::test_bergen_summarizes - Failed: Fixture "bergen...
1 failed


A test that calls a fixture treats it like the helper function it used to be. A fixture is for pytest
to call: `@pytest.fixture` replaced the function with one that refuses to be called by anything
else. Request it by name:


In [30]:
%%writefile scratch/stations/tests/test_direct.py
import pytest

from readings import summarize


@pytest.fixture
def bergen_lines():
    return ["Bergen,4.2", "Bergen,5.8"]


def test_bergen_summarizes(bergen_lines):
    assert summarize(bergen_lines) == {"Bergen": 5.0}


Overwriting scratch/stations/tests/test_direct.py


In [31]:
run_pytest("tests/test_direct.py", "-q")


.                                                                        [100%]
1 passed


### fixture 'tuesday_fiel' not found


In [32]:
%%writefile scratch/stations/tests/test_typo.py
from readings import summarize


def test_the_file_summarizes(tuesday_fiel):
    with open(tuesday_fiel, encoding="utf-8") as file:
        assert summarize(file)["Svalbard"] is None


Writing scratch/stations/tests/test_typo.py


In [33]:
run_pytest("tests/test_typo.py", "-q")


E                                                                        [100%]
==================================== ERRORS ====================================
__________________ ERROR at setup of test_the_file_summarizes __________________
file tests/test_typo.py, line 4
  def test_the_file_summarizes(tuesday_fiel):
E       fixture 'tuesday_fiel' not found
>       available fixtures: cache, capfd, capfdbinary, caplog, capsys, capsysbinary, capteesys, doctest_namespace, monkeypatch, pytestconfig, record_property, record_testsuite_property, record_xml_attribute, recwarn, tmp_path, tmp_path_factory, tmpdir, tmpdir_factory, tuesday, tuesday_file
>       use 'pytest --fixtures [testpath]' for help on them.

tests/test_typo.py:4
=========================== short test summary info ============================
ERROR tests/test_typo.py::test_the_file_summarizes
1 error


An error, not a failure: pytest could not set the test up, so the test never ran. The report lists
every fixture the test could have requested, and `tuesday_file`, from `conftest.py`, is among them,
spelled correctly. An argument name is a request, so a misspelled argument is a request for a fixture
that does not exist:


In [34]:
source = (PROJECT / "tests" / "test_typo.py").read_text()
(PROJECT / "tests" / "test_typo.py").write_text(source.replace("tuesday_fiel", "tuesday_file"))

run_pytest("tests/test_typo.py", "-q")


.                                                                        [100%]
1 passed


### ScopeMismatch: You tried to access the function scoped fixture tmp_path


In [35]:
%%writefile scratch/stations/tests/test_scope.py
import pytest


@pytest.fixture(scope="module")
def shared_folder(tmp_path):
    return tmp_path


def test_the_folder_exists(shared_folder):
    assert shared_folder.is_dir()


Writing scratch/stations/tests/test_scope.py


In [36]:
run_pytest("tests/test_scope.py", "-q")


E                                                                        [100%]
==================================== ERRORS ====================================
___________________ ERROR at setup of test_the_folder_exists ___________________
ScopeMismatch: You tried to access the function scoped fixture tmp_path with a module scoped request object. Requesting fixture stack:
tests/test_scope.py:4:  def shared_folder(tmp_path)
Requested fixture:
_pytest/tmpdir.py:254:  def tmp_path(request: 'FixtureRequest', tmp_path_factory: 'TempPathFactory') -> 'Generator[Path]'
=========================== short test summary info ============================
ERROR tests/test_scope.py::test_the_folder_exists - Failed: ScopeMismatch: Yo...
1 error


`shared_folder` lasts for the whole file, and it requested `tmp_path`, which lasts for one test.
After the first test, its folder would belong to a test that is over, so pytest refuses the request.
A fixture can request fixtures of its own scope or a wider one. `tmp_path_factory`, the
session-scoped fixture behind `tmp_path`, makes a folder that lasts as long as it is needed:


In [37]:
%%writefile scratch/stations/tests/test_scope.py
import pytest


@pytest.fixture(scope="module")
def shared_folder(tmp_path_factory):
    return tmp_path_factory.mktemp("shared")


def test_the_folder_exists(shared_folder):
    assert shared_folder.is_dir()


Overwriting scratch/stations/tests/test_scope.py


In [38]:
run_pytest("tests/test_scope.py", "-q")


.                                                                        [100%]
1 passed


### assert 5 == 4


In [39]:
%%writefile scratch/stations/tests/test_shared.py
import pytest


@pytest.fixture(scope="module")
def stations():
    return ["Bergen", "Oslo", "Svalbard", "Tromso"]


def test_a_new_station_can_be_added(stations):
    stations.append("Alta")
    assert "Alta" in stations


def test_there_are_four_stations(stations):
    assert len(stations) == 4


Writing scratch/stations/tests/test_shared.py


In [40]:
run_pytest("tests/test_shared.py", "-q")


.F                                                                       [100%]
=================================== FAILURES ===================================
_________________________ test_there_are_four_stations _________________________

stations = ['Bergen', 'Oslo', 'Svalbard', 'Tromso', 'Alta']

    def test_there_are_four_stations(stations):
>       assert len(stations) == 4
E       AssertionError: assert 5 == 4
E        +  where 5 = len(['Bergen', 'Oslo', 'Svalbard', 'Tromso', 'Alta'])

tests/test_shared.py:15: AssertionError
=========================== short test summary info ============================
FAILED tests/test_shared.py::test_there_are_four_stations - AssertionError: a...
1 failed, 1 passed


`stations` is built once for the file, so both tests received the same list, and the first test's
`append` was still in it when the second test ran, as `stations = [..., 'Alta']` in the report
shows. Run alone, the second test passes, which makes this failure hard to trace. A test that
changes a value needs a value of its own, which the default scope gives it:


In [41]:
source = (PROJECT / "tests" / "test_shared.py").read_text()
(PROJECT / "tests" / "test_shared.py").write_text(source.replace('@pytest.fixture(scope="module")', "@pytest.fixture"))

run_pytest("tests/test_shared.py", "-q")


..                                                                       [100%]
2 passed


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
module, the script, and every test file:


In [42]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A fixture is a function marked with `@pytest.fixture`. A test requests it by naming it as an
  argument, and pytest runs it first and passes in what it returned.
- In a fixture that uses `yield`, the code after the `yield` is cleanup, and pytest runs it after the
  test, whether the test passed or failed.
- A fixture can request fixtures, and `tmp_path` gives every test a new, empty folder.
- `conftest.py` shares its fixtures with every test in its folder and the folders below, with no
  import.
- Scope sets how long a value lasts: one test by default, or a class, a module, a package or the
  whole run, and a test must not change a value it shares.
- `capsys` captures what the code under test prints, and `monkeypatch.setenv` sets an environment
  variable for one test.
- `--setup-show` shows the fixtures of every test, set up and torn down in order.


## What is next

The **Parametrize** notebook runs one test over many inputs, with a result reported for each input,
and gives a fixture a list of values, so that every test that requests it runs once for each value.


---

&#8592; **Previous:** [Test Structure](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/03-test-structure.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
